# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazama-tech/Flyrank-Ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import duckdb
import pandas as pd


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

print("Connected successfully!")

Connected successfully!


**Unit of analysis**

One row represents the performance of one content item for one client on one report date.

**Tables**

I will use fact_content_daily_performance for daily performance signals and dim_content for content-level attributes where needed.

**Time window**

I will use March 2026 as the development window and keep June 2026 as the sealed final month.

**Prediction**

I want to predict whether a piece of content will become a high-performing "hit", using information available before the prediction point.

**Deliberate exclusion**

I will exclude future outcome variables and target-derived fields because they would not be knowable at the decision moment and could cause leakage.

In [4]:
# This cell is for CODE (numbers, a query, a check).
df = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 5
""").df()

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   report_date               5 non-null      datetime64[us]
 1   client_hash_id            5 non-null      object        
 2   content_hash_id           5 non-null      object        
 3   client_has_gsc            5 non-null      bool          
 4   client_has_ga4            5 non-null      bool          
 5   gsc_data_available        5 non-null      bool          
 6   ga4_data_available        5 non-null      bool          
 7   gsc_impressions           5 non-null      int64         
 8   gsc_clicks                5 non-null      int64         
 9   gsc_sum_position          5 non-null      int64         
 10  gsc_avg_position          5 non-null      float64       
 11  ga4_pageviews             5 non-null      int64         
 12  ga4_sessions              

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# Grain check.
grain_check = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


Grain check: The March 2026 data contains no duplicate combinations of report_date, client_hash_id, and content_hash_id. This supports the stated grain of one row per client, content item, and report date.

In [6]:
#How many rows are in my March slice
row_date_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

row_date_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


Row count and date-span check: The March 2026 slice contains 9,841,378 rows, with dates ranging from 2026-03-01 to 2026-03-31. This confirms that the selected slice covers the intended March 2026 window.

In [7]:
availability_columns = con.sql("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""").df()

availability_columns[
    availability_columns["column_name"].str.contains(
        "available", case=False, na=False
    )
]

,column_name,column_type,null,key,default,extra
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None


In [8]:
availability_check = con.sql("""
    SELECT
        COUNT(*) AS available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


3,611,061 rows have gsc_data_available IS TRUE.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
# Build the March feature frame.
# One row = one client + one content item for March.

feature_frame = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature 1: total search visibility during March
        SUM(gsc_impressions) AS march_gsc_impressions,

        -- Feature 2: total search clicks during March
        SUM(gsc_clicks) AS march_gsc_clicks,

        -- Feature 3: March search position, weighted by impressions
        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0)
            AS march_gsc_avg_position,

        -- Feature 4: total GA4 sessions during March
        SUM(ga4_sessions) AS march_ga4_sessions,

        -- Feature 5: total scroll events during March
        SUM(scroll_events) AS march_scroll_events

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )

    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    ORDER BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)

display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)


,client_hash_id,content_hash_id,march_gsc_impressions,march_gsc_clicks,march_gsc_avg_position,march_ga4_sessions,march_scroll_events
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1.0,0.0,9.000000,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331.0,2.0,14.377644,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33.0,0.0,9.363636,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145.0,0.0,8.124138,NaN,NaN
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461.0,0.0,14.488069,NaN,NaN


In [15]:
# Confirm the feature frame has one row per client-content pair.
duplicate_features = feature_frame.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print("Duplicate client-content rows:", duplicate_features)
print("Feature columns:", feature_frame.columns.tolist())

Duplicate client-content rows: 0
Feature columns: ['client_hash_id', 'content_hash_id', 'march_gsc_impressions', 'march_gsc_clicks', 'march_gsc_avg_position', 'march_ga4_sessions', 'march_scroll_events']


### Feature-frame check

The March feature frame contains 176,738 unique client-content pairs and five model features. The duplicate check returned 0, confirming that each row represents one client-content pair for the March feature window. The features are based only on information observed during March and do not use the future outcome.

### Why these five features are valid

- **`march_gsc_impressions`** — Knowable at the decision moment because it records the search impressions observed during March.
- **`march_gsc_clicks`** — Knowable at the decision moment because it records the search clicks observed during March.
- **`march_gsc_avg_position`** — Knowable at the decision moment because it summarizes the content's observed search position during March.
- **`march_ga4_sessions`** — Knowable at the decision moment because it records sessions observed during March.
- **`march_scroll_events`** — Knowable at the decision moment because it records page engagement observed during March.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Leakage trap

The prediction target is whether a content item becomes a high-performing "hit" in April 2026. I define a hit as a content item whose April GSC impressions are in the top 10% among content with available GSC data.

The five March features are the honest predictors. I will first measure their quick score, then deliberately add one label-derived feature to demonstrate how leakage can produce an artificially strong score.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Build the April outcome at the same client-content level.

april_outcome = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_gsc_impressions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )

    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("April outcome shape:", april_outcome.shape)
display(april_outcome.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April outcome shape: (194760, 3)


,client_hash_id,content_hash_id,april_gsc_impressions
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0


In [19]:
# Define the April hit threshold: top 10% of April impressions.

hit_threshold = april_outcome["april_gsc_impressions"].quantile(0.90)

april_outcome["is_hit"] = (
    april_outcome["april_gsc_impressions"] >= hit_threshold
).astype(int)

print("April hit threshold:", hit_threshold)
print("\nHit distribution:")
print(april_outcome["is_hit"].value_counts())
print("\nHit rate:")
print(april_outcome["is_hit"].mean())

April hit threshold: 3386.0

Hit distribution:
is_hit
0    175283
1     19477
Name: count, dtype: int64

Hit rate:
0.10000513452454303


In [20]:
# Combine March features with the April outcome.

model_df = feature_frame.merge(
    april_outcome[
        [
            "client_hash_id",
            "content_hash_id",
            "is_hit"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model dataset shape:", model_df.shape)
print("\nLabel distribution:")
print(model_df["is_hit"].value_counts())

Model dataset shape: (158549, 8)

Label distribution:
is_hit
0    139523
1     19026
Name: count, dtype: int64


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

In [22]:
# The five honest March features.

honest_features = [
    "march_gsc_impressions",
    "march_gsc_clicks",
    "march_gsc_avg_position",
    "march_ga4_sessions",
    "march_scroll_events"
]

X = model_df[honest_features]
y = model_df["is_hit"]

# Split the data while preserving the hit rate.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Simple pipeline: fill missing feature values, then fit a small tree.
honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    )
)

honest_model.fit(X_train, y_train)

honest_probability = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(
    y_test,
    honest_probability
)

print("Honest ROC-AUC:", round(honest_auc, 4))

Honest ROC-AUC: 0.9639


### Deliberate leakage experiment

To demonstrate leakage, I will temporarily add `april_gsc_impressions` as a feature. This value is derived from the same future-month outcome used to create `is_hit`, so it would not be available at the March decision moment.

This feature is intentionally invalid and is included only to demonstrate how leakage can make model performance appear unrealistically strong.

In [24]:
# Remove the deliberately leaked future feature.
# The final feature set contains only information available at the March decision moment.

final_features = honest_features.copy()

print("Final features:")
for feature in final_features:
    print("-", feature)

print("\nHonest ROC-AUC to keep:", round(honest_auc, 4))

Final features:
- march_gsc_impressions
- march_gsc_clicks
- march_gsc_avg_position
- march_ga4_sessions
- march_scroll_events

Honest ROC-AUC to keep: 0.9639


### Leakage lesson

The honest model uses only five March features and achieved a ROC-AUC of 0.9639.

When `april_gsc_impressions` was deliberately added, the ROC-AUC increased to 1.00. This feature is invalid because April impressions are part of the future outcome used to define `is_hit`.

The inflated score demonstrates target leakage: information derived from the label was allowed into the feature set. The leaked feature was removed, and the honest ROC-AUC was retained as the valid result.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.